# 03 - Exploratory Analysis

This notebook begins the exploratory analysis of the New Orleans subset.

The goal is to understand whether the filtered data is suitable for the two main technical directions:

- social network analysis,
- time series forecasting.

In [1]:
from pathlib import Path
import csv
import json
from collections import Counter
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_YELP_DIR = DATA_DIR / "raw" / "yelp"
INTERIM_DIR = DATA_DIR / "interim"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

BUSINESS_PATH = RAW_YELP_DIR / "yelp_academic_dataset_business.json"
REVIEW_PATH = RAW_YELP_DIR / "yelp_academic_dataset_review.json"
USER_PATH = RAW_YELP_DIR / "yelp_academic_dataset_user.json"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp


In [2]:
CITY_INTERIM_DIR = INTERIM_DIR / "new_orleans"
BUSINESSES_PATH = CITY_INTERIM_DIR / "businesses.jsonl"
REVIEWS_PATH = CITY_INTERIM_DIR / "reviews.jsonl"
USERS_PATH = CITY_INTERIM_DIR / "users.jsonl"
SUMMARY_PATH = CITY_INTERIM_DIR / "summary.json"

with SUMMARY_PATH.open("r", encoding="utf-8") as file:
    summary = json.load(file)

summary

{'business_count': 6215,
 'created_at_utc': '2026-05-12T17:30:52.351011+00:00',
 'dataset_slug': 'new_orleans',
 'matched_user_profile_count': 245419,
 'max_review_date': '2022-01-19 19:47:59',
 'min_review_date': '2005-03-14 18:07:51',
 'outputs': {'businesses': 'C:\\Users\\mehdi\\OneDrive\\Documents\\community-forecasting-yelp\\data\\interim\\new_orleans\\businesses.jsonl',
             'reviews': 'C:\\Users\\mehdi\\OneDrive\\Documents\\community-forecasting-yelp\\data\\interim\\new_orleans\\reviews.jsonl',
             'users': 'C:\\Users\\mehdi\\OneDrive\\Documents\\community-forecasting-yelp\\data\\interim\\new_orleans\\users.jsonl'},
 'review_count': 635521,
 'target_city': 'New Orleans',
 'target_state': 'LA',
 'unique_reviewing_user_count': 245421}

## First EDA Questions

The next analysis should answer:

- How many reviews are posted per year and per month?
- Which business categories dominate New Orleans?
- How sparse is review activity across businesses?
- How concentrated is reviewer activity across users?
- Are there enough repeat reviewers and friendship links to build a meaningful social graph?

In [3]:
def iter_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            try:
                yield json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path} at line {line_number}") from exc

In [4]:
review_year_counts = Counter()
review_month_counts = Counter()
business_review_counts = Counter()
user_review_counts = Counter()

for record in iter_jsonl(REVIEWS_PATH):
    date_text = record.get("date", "")
    if len(date_text) >= 7:
        review_year_counts[date_text[:4]] += 1
        review_month_counts[date_text[:7]] += 1
    business_review_counts[record["business_id"]] += 1
    user_review_counts[record["user_id"]] += 1

print("Review counts by year:")
for year, count in sorted(review_year_counts.items()):
    print(f"{year}: {count:,}")

print("\nMost reviewed New Orleans businesses by review count:")
for business_id, count in business_review_counts.most_common(10):
    print(f"{business_id}: {count:,}")

print("\nMost active reviewers in subset:")
for user_id, count in user_review_counts.most_common(10):
    print(f"{user_id}: {count:,}")

Review counts by year:
2005: 291
2006: 298
2007: 1,233
2008: 3,192
2009: 6,721
2010: 14,150
2011: 26,391
2012: 28,906
2013: 38,107
2014: 51,148
2015: 66,758
2016: 70,981
2017: 74,184
2018: 88,584
2019: 85,148
2020: 35,265
2021: 41,942
2022: 2,222

Most reviewed New Orleans businesses by review count:
_ab50qdWOk0DdB6XOrBitw: 7,673
ac1AeYqs8Z4_e2X5M3if2A: 7,516
oBNrLz4EDhiscSlbOl8uAw: 5,264
iSRTaT9WngzB8JJ2YKJUig: 5,254
VQcCL9PiNL_wkGf-uF3fjg: 5,146
_C7QiQQc47AOEv4PE3Kong: 4,969
GBTPC53ZrG1ZBY3DT8Mbcw: 4,661
6a4gLLFSgr-Q6CZXDLzBGQ: 4,480
VaO-VW3e1kARkU9bP1E7Fw: 4,034
qb28j-FNX1_6xm7u372TZA: 3,971

Most active reviewers in subset:
0Igx-a1wAstiBDerGxXk2A: 1,420
Xw7ZjaGfr0WNVt6s_5KZfA: 1,237
1HM81n6n4iPIFU5d2Lokhw: 1,077
CfX4sTIFFNaRchNswqhVfg: 835
E4BsVQnG5zetbwv2x8QIWg: 756
0G-QF457q_0Z_jKqh6xWiA: 548
iT1AA74QFn_a0Xq9-ggSGA: 461
5tXRxr4T24Awl7vjyCvIcQ: 440
riWDgbauId8TK7YFVgMNJw: 418
Ase_kJIYuT6yOsqqVPuWUA: 417


In [5]:
category_counts = Counter()

for record in iter_jsonl(BUSINESSES_PATH):
    categories = record.get("categories") or ""
    for category in categories.split(","):
        category = category.strip()
        if category:
            category_counts[category] += 1

print("Top business categories:")
for category, count in category_counts.most_common(25):
    print(f"{category:<35} {count:>6}")

Top business categories:
Restaurants                           2261
Food                                  1263
Shopping                              1028
Nightlife                              942
Bars                                   826
Event Planning & Services              656
Hotels & Travel                        620
Arts & Entertainment                   539
Beauty & Spas                          437
Cajun/Creole                           402
Breakfast & Brunch                     367
Sandwiches                             354
Seafood                                332
American (New)                         324
Local Services                         322
Coffee & Tea                           318
Fashion                                312
Active Life                            302
American (Traditional)                 299
Hotels                                 296
Home Services                          282
Health & Medical                       277
Automotive                   

## Interpretation: Dataset Size and Coverage

The New Orleans subset is large enough for both parts of the project. It contains **6,215 businesses**, **635,521 reviews**, and **245,421 reviewing users**, covering reviews from **March 2005 to January 2022**. This gives a long temporal window for monthly forecasting and a large user base for social network construction.

The subset is still much smaller than the full Yelp dataset, which makes it practical for iterative academic analysis. This supports the methodological choice to focus on one city rather than trying to model the entire dataset at once.

## Additional EDA Statistics

The next cell summarizes concentration and sparsity. These are important because the project needs to decide which businesses and users are suitable for forecasting and network analysis.

In [6]:
from statistics import mean, median

business_review_values = list(business_review_counts.values())
user_review_values = list(user_review_counts.values())

def count_at_least(values, threshold):
    return sum(value >= threshold for value in values)

print("Business review distribution")
print(f"Businesses with reviews: {len(business_review_values):,}")
print(f"Mean reviews per business: {mean(business_review_values):.2f}")
print(f"Median reviews per business: {median(business_review_values):,.0f}")
print(f"Maximum reviews for one business: {max(business_review_values):,}")
for threshold in [10, 25, 50, 100, 250, 500, 1000]:
    print(f"Businesses with at least {threshold:>4} reviews: {count_at_least(business_review_values, threshold):,}")

review_total = sum(business_review_values)
top_10_business_reviews = sum(count for _, count in business_review_counts.most_common(10))
top_100_business_reviews = sum(count for _, count in business_review_counts.most_common(100))
print(f"Top 10 businesses' share of reviews: {top_10_business_reviews / review_total:.2%}")
print(f"Top 100 businesses' share of reviews: {top_100_business_reviews / review_total:.2%}")

print("\nUser review distribution")
print(f"Reviewing users: {len(user_review_values):,}")
print(f"Mean reviews per user: {mean(user_review_values):.2f}")
print(f"Median reviews per user: {median(user_review_values):,.0f}")
print(f"Maximum reviews by one user: {max(user_review_values):,}")
for threshold in [2, 3, 5, 10, 25, 50, 100]:
    print(f"Users with at least {threshold:>3} reviews: {count_at_least(user_review_values, threshold):,}")

top_10_user_reviews = sum(count for _, count in user_review_counts.most_common(10))
top_100_user_reviews = sum(count for _, count in user_review_counts.most_common(100))
print(f"Top 10 users' share of reviews: {top_10_user_reviews / review_total:.2%}")
print(f"Top 100 users' share of reviews: {top_100_user_reviews / review_total:.2%}")

Business review distribution
Businesses with reviews: 6,215
Mean reviews per business: 102.26
Median reviews per business: 22
Maximum reviews for one business: 7,673
Businesses with at least   10 reviews: 4,649
Businesses with at least   25 reviews: 2,952
Businesses with at least   50 reviews: 1,962
Businesses with at least  100 reviews: 1,225
Businesses with at least  250 reviews: 547
Businesses with at least  500 reviews: 259
Businesses with at least 1000 reviews: 89
Top 10 businesses' share of reviews: 8.33%
Top 100 businesses' share of reviews: 31.46%

User review distribution
Reviewing users: 245,421
Mean reviews per user: 2.59
Median reviews per user: 1
Maximum reviews by one user: 1,420
Users with at least   2 reviews: 96,699
Users with at least   3 reviews: 56,380
Users with at least   5 reviews: 26,598
Users with at least  10 reviews: 8,204
Users with at least  25 reviews: 1,666
Users with at least  50 reviews: 585
Users with at least 100 reviews: 180
Top 10 users' share of re

## Interpretation: Concentration and Sparsity

Review activity is highly uneven across businesses. The average business has about **102 reviews**, but the median is only **22**, meaning a small number of very popular businesses pull the average upward. The top 10 businesses account for about **8.33%** of all New Orleans reviews, and the top 100 account for about **31.46%**.

This matters for forecasting. A model trained on all businesses would mix very active businesses with sparse businesses that have weak or irregular time series. A more defensible first forecasting target is therefore monthly review activity for businesses above a minimum historical activity threshold, such as **at least 50 or 100 reviews**.

User activity is also sparse. The median user wrote only **1** review in the New Orleans subset, while a small group of highly active users wrote hundreds or more. This suggests that social network analysis should not treat all users as equally informative. For influence analysis, repeat reviewers and users with friendship links are likely to be more meaningful than one-time reviewers.

In [7]:
months = sorted(review_month_counts)
pre_2020_months = [month for month in months if month < "2020-01"]
post_2020_months = [month for month in months if "2020-01" <= month < "2022-01"]
peak_month, peak_count = max(review_month_counts.items(), key=lambda item: item[1])

print(f"Observed months: {len(months)}")
print(f"First observed month: {months[0]}")
print(f"Last observed month: {months[-1]}")
print(f"Average monthly reviews before 2020: {mean(review_month_counts[m] for m in pre_2020_months):,.2f}")
print(f"Average monthly reviews in 2020-2021: {mean(review_month_counts[m] for m in post_2020_months):,.2f}")
print(f"Peak month: {peak_month} with {peak_count:,} reviews")

Observed months: 202
First observed month: 2005-03
Last observed month: 2022-01
Average monthly reviews before 2020: 3,141.76
Average monthly reviews in 2020-2021: 3,216.96
Peak month: 2019-03 with 9,396 reviews


## Interpretation: Temporal Pattern

The review history spans **202 monthly observations**, from **2005-03** to **2022-01**. This is enough for monthly time series analysis.

The yearly counts show strong growth from 2005 through the late 2010s, peaking around **2018-2019**. There is a clear drop in **2020**, which is likely connected to COVID-19 disruption in restaurants, tourism, nightlife, and local services. This is especially relevant for New Orleans because many dominant categories are hospitality-related.

For forecasting, this means the time series is not stationary over the whole period. We should be careful with models that assume stable historical behavior. A sensible approach is to compare a simple baseline with models that use recent history, seasonality, and possibly a COVID-era indicator or train/test split that acknowledges the structural break.

## Interpretation: Business Categories

The category distribution strongly reflects the local economy and Yelp usage context in New Orleans. The most common categories include **Restaurants**, **Food**, **Nightlife**, **Bars**, **Hotels & Travel**, **Cajun/Creole**, **Seafood**, and **Local Flavor**.

This supports the project theme because community activity is not random: review behavior is likely shaped by tourism, dining, nightlife, and local events. It also suggests that category features may be useful in forecasting. For example, restaurants and nightlife businesses may have stronger seasonality and stronger COVID-era disruption than categories such as home services or health services.

For social network analysis, the category mix also matters. Users who review restaurants, bars, and tourism-related businesses may form different behavioral communities from users reviewing local services or shopping. Later, we can examine whether detected user communities align with business categories or review patterns.

## Visual Evidence for the EDA Conclusions

The following plots support the main exploratory findings: long-term growth followed by disruption, strong concentration of reviews across businesses, sparse activity across users, a hospitality-heavy category mix, and a rating distribution skewed toward positive reviews.

Figures are saved in:

```text
outputs/figures/eda/
```

In [8]:
import matplotlib.pyplot as plt
from datetime import datetime

FIGURES_DIR = OUTPUTS_DIR / "figures" / "eda"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

figure_paths = {}

### Review Activity Over Time

The yearly and monthly plots show that New Orleans Yelp activity grew substantially from the mid-2000s through the late 2010s, followed by a visible decline around 2020. This supports the interpretation that the time series contains both trend and disruption rather than a stable stationary pattern.

In [9]:
years = sorted(review_year_counts)
year_values = [review_year_counts[year] for year in years]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(years, year_values, color="#4C78A8")
ax.set_title("New Orleans Yelp Reviews by Year")
ax.set_xlabel("Year")
ax.set_ylabel("Review count")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()

path = FIGURES_DIR / "reviews_by_year.png"
fig.savefig(path, dpi=160)
plt.show()
plt.close(fig)
figure_paths["reviews_by_year"] = path
print(path)

C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\outputs\figures\eda\reviews_by_year.png


In [10]:
month_items = sorted(
    (datetime.strptime(month, "%Y-%m"), count)
    for month, count in review_month_counts.items()
)
month_dates = [item[0] for item in month_items]
month_values = [item[1] for item in month_items]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(month_dates, month_values, color="#2F855A", linewidth=1.6)
ax.axvspan(datetime(2020, 1, 1), datetime(2021, 12, 31), color="#E45756", alpha=0.12, label="2020-2021")
ax.set_title("Monthly Review Activity in New Orleans")
ax.set_xlabel("Month")
ax.set_ylabel("Review count")
ax.legend(frameon=False)
fig.tight_layout()

path = FIGURES_DIR / "monthly_review_activity.png"
fig.savefig(path, dpi=160)
plt.show()
plt.close(fig)
figure_paths["monthly_review_activity"] = path
print(path)

C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\outputs\figures\eda\monthly_review_activity.png


### Business Review Concentration

The distribution plot and concentration curve show why business-level forecasting should use an activity threshold. Most businesses have relatively few reviews, while a smaller group of highly visible businesses receives a large share of the total activity.

In [11]:
business_counts_sorted = sorted(business_review_values, reverse=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(business_review_values, bins=60, color="#F58518", edgecolor="white")
ax.set_yscale("log")
ax.set_title("Distribution of Reviews per Business")
ax.set_xlabel("Reviews per business")
ax.set_ylabel("Number of businesses, log scale")
fig.tight_layout()

path = FIGURES_DIR / "business_review_distribution.png"
fig.savefig(path, dpi=160)
plt.show()
plt.close(fig)
figure_paths["business_review_distribution"] = path
print(path)

C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\outputs\figures\eda\business_review_distribution.png


In [12]:
total_business_reviews = sum(business_counts_sorted)
cumulative_reviews = []
running_total = 0
for count in business_counts_sorted:
    running_total += count
    cumulative_reviews.append(running_total / total_business_reviews * 100)
percent_businesses = [(index + 1) / len(business_counts_sorted) * 100 for index in range(len(business_counts_sorted))]

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(percent_businesses, cumulative_reviews, color="#B279A2", linewidth=2)
ax.plot([0, 100], [0, 100], color="#999999", linestyle="--", linewidth=1, label="Equal distribution")
ax.set_title("Business Review Concentration Curve")
ax.set_xlabel("Top businesses included (%)")
ax.set_ylabel("Cumulative reviews captured (%)")
ax.legend(frameon=False)
fig.tight_layout()

path = FIGURES_DIR / "business_review_concentration.png"
fig.savefig(path, dpi=160)
plt.show()
plt.close(fig)
figure_paths["business_review_concentration"] = path
print(path)

C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\outputs\figures\eda\business_review_concentration.png


### User Activity Sparsity

The user distribution shows that many users contribute only once in the New Orleans subset. This supports using repeat reviewers and users with meaningful social connections for influence analysis, rather than assuming every user contributes equally to the social graph.

In [13]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(user_review_values, bins=60, color="#72B7B2", edgecolor="white")
ax.set_yscale("log")
ax.set_title("Distribution of Reviews per User")
ax.set_xlabel("Reviews per user in New Orleans subset")
ax.set_ylabel("Number of users, log scale")
fig.tight_layout()

path = FIGURES_DIR / "user_review_distribution.png"
fig.savefig(path, dpi=160)
plt.show()
plt.close(fig)
figure_paths["user_review_distribution"] = path
print(path)

C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\outputs\figures\eda\user_review_distribution.png


### Category Mix

The category plot confirms that the subset is strongly shaped by restaurants, food, bars, nightlife, hotels, travel, and local entertainment. This supports later use of category features and helps explain why seasonal and COVID-era effects may matter.

In [14]:
top_categories = category_counts.most_common(15)
categories = [category for category, _ in reversed(top_categories)]
category_values = [count for _, count in reversed(top_categories)]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(categories, category_values, color="#54A24B")
ax.set_title("Top Business Categories in New Orleans Subset")
ax.set_xlabel("Business count")
ax.set_ylabel("Category")
fig.tight_layout()

path = FIGURES_DIR / "top_business_categories.png"
fig.savefig(path, dpi=160)
plt.show()
plt.close(fig)
figure_paths["top_business_categories"] = path
print(path)

C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\outputs\figures\eda\top_business_categories.png


### Rating Distribution

The rating distribution is skewed toward positive reviews, especially 4- and 5-star reviews. This is useful context for interpretation: review volume is the main forecasting target for now, but review sentiment or rating mix could later become an additional signal of business popularity or community response.

In [15]:
star_counts = Counter()
for record in iter_jsonl(REVIEWS_PATH):
    star_counts[int(record["stars"])] += 1

stars = sorted(star_counts)
star_values = [star_counts[star] for star in stars]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([str(star) for star in stars], star_values, color="#ECA82C")
ax.set_title("Review Rating Distribution")
ax.set_xlabel("Stars")
ax.set_ylabel("Review count")
fig.tight_layout()

path = FIGURES_DIR / "rating_distribution.png"
fig.savefig(path, dpi=160)
plt.show()
plt.close(fig)
figure_paths["rating_distribution"] = path
print(path)

C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\outputs\figures\eda\rating_distribution.png


### Figure Summary

The plots reinforce the main EDA conclusions:

- The dataset has enough monthly history for forecasting.
- Review activity grew over time but was disrupted around 2020.
- Business review activity is concentrated, so forecasting should start with sufficiently active businesses.
- User activity is sparse, so social influence features should focus on repeat reviewers and connected users.
- The category mix is strongly hospitality-oriented, which is important for interpreting seasonality and disruption.
- Ratings are skewed positive, so review volume and engagement may be more informative initial targets than average star rating alone.

## Detailed Analysis of EDA Results

This section connects the numerical summaries and plots to the project's methodological choices. The goal is not only to describe the dataset, but to decide what can be modeled responsibly in the later social network and time series stages.

### 1. Review Activity Over Time

The yearly review plot shows a long growth phase followed by a sharp disruption. Review volume is very low in the earliest years, with only **291 reviews in 2005** and **298 in 2006**, then grows steadily as Yelp adoption increases. By the mid-to-late 2010s, the platform has become much more active in New Orleans: review volume reaches **88,584 reviews in 2018**, the highest full year in the subset.

The decline in 2020 is substantial. Reviews fall from **85,148 in 2019** to **35,265 in 2020**, a decrease of about **58.6%**. Compared with 2018, the 2020 volume is about **60.2% lower**. The partial recovery in 2021, with **41,942 reviews**, suggests that activity begins to return, but it does not reach the pre-2020 level.

This is important for forecasting because the review series is not just seasonal noise around a stable average. It contains adoption growth, maturity, and a major external shock. A model trained blindly on the full history may overestimate post-2020 activity or underestimate pre-2020 growth. Later forecasting experiments should therefore use time-aware train/test splits and consider whether 2020 should be treated as a structural break.

### 2. Monthly Review Activity

The monthly plot confirms that the dataset is suitable for time series forecasting. There are **202 monthly observations**, from **2005-03** to **2022-01**. This is enough history for baseline models, seasonal models, and rolling validation.

The monthly averages also show clear regime differences. Before 2010, the subset averages only about **206 reviews per month**, reflecting early platform adoption. From 2010 to 2014, the average rises to about **2,645 reviews per month**. From 2015 to 2019, the average reaches about **6,428 reviews per month**, with the peak month being **2019-03**, at **9,396 reviews**. During 2020-2021, the average falls to about **3,217 reviews per month**.

This supports using monthly review count as the first forecasting target. Monthly aggregation is detailed enough to capture seasonality and shocks, but not so granular that the series becomes too sparse. Daily forecasting would likely be noisier, while yearly forecasting would hide seasonal and event-driven patterns.

### 3. Business Review Distribution

The business review distribution is highly skewed. The mean business has **102.26 reviews**, but the median business has only **22 reviews**. This large gap means that a small number of highly reviewed businesses pull the average upward.

The threshold counts make the sparsity problem concrete:

- **74.8%** of businesses have at least 10 reviews.
- **47.5%** have at least 25 reviews.
- **31.6%** have at least 50 reviews.
- **19.7%** have at least 100 reviews.
- Only **1.4%** have at least 1,000 reviews.

This matters because forecasting requires enough historical observations. A business with only a handful of reviews over 17 years will not produce a reliable monthly time series. Therefore, the forecasting stage should not start by modeling every business. A reasonable first design is to forecast businesses above a minimum activity threshold, such as **50 or 100 total reviews**, and then discuss the trade-off between coverage and reliability.

### 4. Business Review Concentration

The concentration curve shows that review activity is not evenly distributed across businesses. The top **10 businesses** account for about **8.33%** of all reviews, the top **100 businesses** account for about **31.46%**, and the top **500 businesses** account for about **61.72%**.

The most reviewed businesses are recognizable hospitality and food destinations, including **Acme Oyster House**, **Oceana Grill**, **Ruby Slipper**, **Mother's Restaurant**, **Royal House**, and **Commander's Palace**. This is consistent with the category distribution: New Orleans Yelp activity is strongly shaped by restaurants, tourism, and nightlife.

For the project, this suggests that popularity forecasting should account for business heterogeneity. A single global model may be useful, but the model should include business-level features such as category, historical review volume, average rating, and possibly network-derived reviewer features. It also suggests that predicting trends for already-popular businesses may be easier than predicting sudden growth for low-activity businesses.

### 5. User Activity Distribution

The user activity distribution is even more sparse than the business distribution. The average user wrote **2.59 reviews** in the New Orleans subset, but the median user wrote only **1 review**. This means most users appear only once and provide limited information for user-level behavior modeling.

Only **39.4%** of users wrote at least 2 reviews, **10.8%** wrote at least 5, and just **3.34%** wrote at least 10. At the extreme, a small number of highly active users wrote hundreds or more reviews, with the maximum being **1,420 reviews**.

This has direct implications for social network analysis. If we build a user graph, many nodes may be weakly informative or disconnected from meaningful local activity. Influence analysis should therefore focus on users who are active enough to have observable behavior, and especially on users who also have friendship links inside the selected subset. One-time reviewers can still contribute to business popularity counts, but they are less useful for detecting communities or estimating influence.

### 6. Category Distribution

The category plot confirms that New Orleans Yelp activity is dominated by hospitality and local experience categories. **Restaurants** appear in **36.38%** of businesses, **Food** in **20.32%**, **Nightlife** in **15.16%**, **Bars** in **13.29%**, and **Hotels & Travel** in **9.98%**. More local categories such as **Cajun/Creole**, **Seafood**, and **Local Flavor** also appear prominently.

This is not just descriptive context; it affects modeling. Businesses in tourism, food, and nightlife are likely to experience stronger seasonality, event effects, and COVID-era disruption than businesses in less socially dependent categories. Category should therefore be considered as an explanatory feature in forecasting and as an interpretation dimension in the final report.

For social network analysis, category distribution can also help explain communities. Groups of users may cluster around restaurant reviewing, nightlife, local services, or tourism-oriented businesses. Later, if community detection reveals groups of users, we can interpret those communities by examining the categories they review most often.

### 7. Rating Distribution

The rating plot shows a strong positive skew. **5-star reviews represent 48.84%** of all reviews, and **4-star reviews represent 23.63%**. Together, 4- and 5-star reviews account for about **72.48%** of the dataset. By contrast, 1- and 2-star reviews account for about **16.90%**.

This suggests that average rating alone may not be the best initial forecasting target. Many businesses may have relatively high ratings, and rating variation may be less informative than activity volume. For the first forecasting stage, review count is a clearer measure of engagement and popularity. Later, rating distribution or sentiment could be added as a complementary signal, for example to distinguish between growing positive attention and growing negative attention.

### 8. Overall EDA Implications for the Project

The EDA supports the project direction, but it also defines constraints. The dataset is large enough for meaningful modeling, but it is not uniform. Review activity varies dramatically across businesses, user activity is sparse, and the time series contains a major disruption around 2020.

The most defensible next steps are:

- Use **monthly review counts** as the first time series target.
- Apply a business activity threshold before forecasting, probably starting with **50 or 100 reviews**.
- Compare simple baselines against models with richer features.
- Treat 2020 as a structural break or special period during evaluation.
- Build the social graph using Yelp friendship data, but analyze whether friendships overlap meaningfully with New Orleans reviewing activity.
- Derive network features for active users and aggregate them to the business-month level.
- Use category and historical popularity as core explanatory variables.

In academic terms, the EDA justifies the methodological design: we should combine temporal patterns, business metadata, and network-derived user features, but we should do so carefully because the data is sparse, skewed, and affected by external shocks.

## EDA Conclusion

The New Orleans subset is suitable for the project. It is large, temporally rich, and domain-relevant. However, the data is also skewed and sparse, which affects the modeling strategy.

Methodological implications:

- Use monthly review counts as the initial time series target.
- Apply a minimum activity threshold before business-level forecasting.
- Keep simple baseline models for comparison.
- Treat 2020 as a potential structural break.
- Use repeat reviewers and friendship data carefully when building social network features.
- Consider business category as an explanatory variable for both forecasting and interpretation.